# VL01 확장 검증 — 추가 학습 없음
`vl01_photo_eval.zip`(약 2GB)을 비공개 Kaggle Dataset으로 업로드 후 이 노트북에 연결하세요.
GPU T4 / Internet ON. 기존 파일럿 Dataset·resume ZIP은 필요 없습니다.

- fixed/photo의 같은 epoch 5 가중치, 동일한 VL01 6,183장, threshold .5.
- 파일럿 학습의 사진·개체·그룹과 겹침 0. 이전 파일럿 검증 394장은 제외.
- 6,026장은 기존 검증에 등장한 그룹의 다른 사진: **개발 검증이며 독립 외부 시험이 아닙니다.**
- 근접 중복은 아직 검사하지 않았습니다. holdout은 사용하지 않습니다.
- clean / Gaussian sigma2(kernel13) / crop을 오른쪽·아래로 각 변의 20% 이동(경계 제한).
- 위치 이동은 합성 조건 하나이며 실제 사용자 분포 전체를 대변하지 않습니다.
- 최종 결과 `vl01_photo_eval_results.zip`을 다운로드하세요.


In [ ]:
from pathlib import Path
import subprocess, sys, zipfile, json, hashlib
subprocess.run([sys.executable, '-m', 'pip', 'install', 'timm==1.0.29'], check=True)
roots = list(Path('/kaggle/input').rglob('eval_manifest.parquet'))
if not roots:
    archives = list(Path('/kaggle/input').rglob('vl01_photo_eval.zip'))
    assert len(archives) == 1, 'vl01_photo_eval.zip Dataset 하나를 연결하세요'
    DATA = Path('/kaggle/temp/vl01_photo_eval')
    DATA.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(archives[0]) as z:
        for name in z.namelist():
            assert not Path(name).is_absolute() and '..' not in Path(name).parts
        z.extractall(DATA)
else:
    assert len(roots) == 1, '평가 Dataset 하나만 연결하세요'
    DATA = roots[0].parent
meta=json.loads((DATA/'package.json').read_text())
for name, digest in meta['files_sha256'].items():
    assert hashlib.sha256((DATA/name).read_bytes()).hexdigest() == digest, name
OUT=Path('/kaggle/working/vl01_photo_eval')
probe="""import torch,timm
assert torch.cuda.is_available(), 'GPU T4를 선택하세요'
m=timm.create_model('tf_efficientnetv2_s.in21k_ft_in1k',pretrained=False,num_classes=2).cuda().eval()
with torch.inference_mode(), torch.autocast('cuda'):
    m(torch.randn(2,3,64,64,device='cuda'))
torch.cuda.synchronize()
print('GPU ready:',torch.cuda.get_device_name(0))
"""
subprocess.run([sys.executable,'-c',probe],check=True)
print('Dataset:',DATA,'Rows:',meta['rows'])


In [ ]:
try:
    subprocess.run([sys.executable, '-u', str(DATA/'tools/eval_vl01_photo.py'),
        '--data', str(DATA), '--out', str(OUT), '--portable', '--batch-size', '32'],
        check=True, cwd=DATA)
finally:
    with zipfile.ZipFile('/kaggle/working/vl01_photo_eval_results.zip','w',zipfile.ZIP_DEFLATED) as z:
        if OUT.exists():
            for path in OUT.rglob('*'):
                if path.is_file():z.write(path,path.relative_to(OUT))
if (OUT/'metrics.csv').exists():print((OUT/'metrics.csv').read_text())
print('Download: vl01_photo_eval_results.zip')
